# BOIA Risk Computation

Compute **class-specific** (per-concept error rate) and **instance-specific**
(average per-concept prediction error) risks for the BOIA dataset using a
trained `BoiaDPL` model, then save them as `.npy` files for curriculum learning.

In [ ]:
import torch
import sys
import numpy as np
import os
from tqdm import tqdm

sys.path.append('../../rss/')

# dataset
from datasets.boia import BOIA
from datasets.utils.sddoia_creation import CONCEPTS_ORDER

# model
from models.boiadpl import BoiaDPL
from argparse import Namespace

args = Namespace(
    dataset="boia",
    task="boia",
    model="boiadpl",
    c_sup=0,
    which_c=[-1],
    backbone="conceptizer",
    batch_size=64,
    lr=0.001,
    weight_decay=0.0001,
    n_epochs=50,
    entropy=False,
    w_sl=10,
    w_h=1,
    w_c=1,
    w_rec=1,
    gamma=1,
    beta=2,
    warmup_steps=0,
    exp_decay=1.0,
    joint=False,
    splitted=False,
    curriculum=False,
    risk_type=None,
    risk_update_freq=None,
    curriculum_steps=5,
    contrastive=False,
    contrastive_temperature=0.5,
    w_contrastive=1.0,
    active_learning=False,
    active_type="random",
    al_cycles=10,
    al_query_size=10,
    seed=0,
    preprocess=False,
    boia_model="ce",
    boia_ood_knowledge=False,
    wandb=None,
    validate=1,
    finetuning=0,
    to_add="",
)

NUM_CONCEPTS = 21  # number of binary concepts in BOIA

## Load Dataset and Model

In [ ]:
# Load BOIA dataset
dataset = BOIA(args)
train_loader, val_loader, test_loader = dataset.get_data_loaders()

# Build model
encoder, _ = dataset.get_backbone()
n_images, c_split = dataset.get_split()

model = BoiaDPL(encoder, n_images=n_images, c_split=c_split, args=args)
model.device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(model.device)

if hasattr(model, "encoder"):
    model.encoder.to(model.device)

# Load checkpoint
model_path = f"../checkpoints/best_model_{args.dataset}_{args.model}_{args.seed}.pth"

if not os.path.exists(model_path):
    print(f"Checkpoint not found: {model_path}")
else:
    print(f"Loading {model_path}...")
    state_dict = torch.load(model_path, map_location=model.device)
    model.load_state_dict(state_dict)
    print("Model loaded successfully.")

model.eval()
print(f"Model on device: {model.device}")

## Concept Labels Reference
Print the concept names and their indices for reference.

In [ ]:
sorted_concepts = sorted(CONCEPTS_ORDER, key=CONCEPTS_ORDER.get)
print("BOIA Concept Labels:")
for i, name in enumerate(sorted_concepts):
    print(f"  [{i:2d}] {name}")

---
# Class-Specific Risk (Per-Concept Error Rate)

For each binary concept $j$, compute the mis-classification rate:

$$R_j = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}[\hat{c}_j^{(i)} \ne c_j^{(i)}]$$

In [ ]:
all_pred_concepts = []
all_true_concepts = []

with torch.no_grad():
    for data in tqdm(train_loader, desc="Computing class-specific risks"):
        images, labels, concepts = data
        images = images.to(model.device)
        concepts = concepts.to(model.device)

        out_dict = model(images)

        # CS contains sigmoid concept probabilities, shape (batch, 21)
        c_probs = out_dict["CS"]
        c_pred = (c_probs >= 0.5).long()

        all_pred_concepts.append(c_pred.cpu().numpy())
        all_true_concepts.append(concepts.cpu().numpy().astype(int))

all_pred_concepts = np.concatenate(all_pred_concepts, axis=0)  # (N, 21)
all_true_concepts = np.concatenate(all_true_concepts, axis=0)  # (N, 21)

# Per-concept error rate
per_concept_risks = (all_pred_concepts != all_true_concepts).astype(float).mean(axis=0)  # (21,)
overall_concept_error = float(per_concept_risks.mean())

print(f"\n{'='*60}")
print(f"Class-Specific (Per-Concept) Risk Results")
print(f"{'='*60}")
for j in range(NUM_CONCEPTS):
    print(f"  Concept {j:2d} ({sorted_concepts[j]:>20s}): error rate = {per_concept_risks[j]:.4f}")
print(f"{'='*60}")
print(f"  Overall concept error: {overall_concept_error:.4f}")

In [ ]:
# Save class-specific risks
save_path_class = f"../class_specific_risks_{args.dataset}_{args.model}.npy"
np.save(save_path_class, per_concept_risks)
print(f"Saved class-specific risks to {save_path_class}")
print(f"Shape: {per_concept_risks.shape}")

## Visualise Class-Specific Risks

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

colors = plt.cm.RdYlGn_r(per_concept_risks / max(per_concept_risks.max(), 1e-8))
bars = ax.bar(range(NUM_CONCEPTS), per_concept_risks, color=colors, edgecolor="black", linewidth=0.5)

ax.set_xticks(range(NUM_CONCEPTS))
ax.set_xticklabels(sorted_concepts, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Error Rate (Risk)")
ax.set_title("BOIA Per-Concept Error Rate (Class-Specific Risk)")
ax.axhline(y=overall_concept_error, color="red", linestyle="--", linewidth=1, label=f"Mean = {overall_concept_error:.4f}")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("../boia_class_specific_risks.png", dpi=300, bbox_inches="tight")
plt.show()

---
# Instance-Specific Risk (Per-Sample Prediction Error)

For each sample $i$, compute the average absolute prediction error across concepts:

$$\text{risk}_i = \frac{1}{C} \sum_{j=1}^{C} |p_j^{(i)} - c_j^{(i)}|$$

where $p_j^{(i)}$ is the model's sigmoid probability for concept $j$.

In [ ]:
from torch.utils.data import DataLoader

# Create a sequential dataloader (shuffle=False) to ensure risks align with dataset indices
sequential_loader = DataLoader(
    dataset.dataset_train,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

instance_risks = []

with torch.no_grad():
    for data in tqdm(sequential_loader, desc="Computing instance-specific risks"):
        images, labels, concepts = data
        images = images.to(model.device)
        concepts = concepts.to(model.device)

        out_dict = model(images)

        # CS contains sigmoid concept probabilities, shape (batch, 21)
        concept_probs = out_dict["CS"]

        # Absolute prediction error per concept
        errors = torch.abs(concept_probs - concepts.float())  # (batch, 21)
        # Mean error across concepts for each sample
        sample_risks = errors.mean(dim=-1)  # (batch,)
        instance_risks.extend(sample_risks.cpu().tolist())

instance_risks = np.array(instance_risks)

print(f"\n{'='*60}")
print(f"Instance-Specific Risk Results")
print(f"{'='*60}")
print(f"  Computed risks for {len(instance_risks)} samples.")
print(f"  Min Risk:  {np.min(instance_risks):.6f}")
print(f"  Max Risk:  {np.max(instance_risks):.6f}")
print(f"  Mean Risk: {np.mean(instance_risks):.6f}")
print(f"  Std Risk:  {np.std(instance_risks):.6f}")

In [ ]:
# Save instance-specific risks
save_path_instance = f"../instance_specific_risks_{args.dataset}_{args.model}.npy"
np.save(save_path_instance, instance_risks)
print(f"Saved instance-specific risks to {save_path_instance}")
print(f"Shape: {instance_risks.shape}")

## Analyse Instance Risks

In [ ]:
# Sort by risk for inspection
sorted_indices = np.argsort(instance_risks)
sorted_risks = instance_risks[sorted_indices]
real_concepts = dataset.dataset_train.real_concepts  # (N, 21)
sorted_concepts_arr = real_concepts[sorted_indices]

print("--- Top 10 Easiest Samples (Lowest Risk) ---")
for i in range(10):
    idx = sorted_indices[i]
    active = [sorted_concepts[c] for c in range(NUM_CONCEPTS) if sorted_concepts_arr[i, c] == 1]
    print(f"  Sample {idx}: Risk = {sorted_risks[i]:.6f}, Active concepts = {active}")

print(f"\n--- Top 10 Hardest Samples (Highest Risk) ---")
for i in range(10):
    idx = sorted_indices[-(i+1)]
    active = [sorted_concepts[c] for c in range(NUM_CONCEPTS) if sorted_concepts_arr[-(i+1), c] == 1]
    print(f"  Sample {idx}: Risk = {sorted_risks[-(i+1)]:.6f}, Active concepts = {active}")

In [ ]:
# Histogram of instance risks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(instance_risks, bins=50, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(x=np.mean(instance_risks), color="red", linestyle="--", label=f"Mean = {np.mean(instance_risks):.4f}")
axes[0].set_xlabel("Instance Risk")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Instance-Specific Risks")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Sorted risk curve
axes[1].plot(sorted_risks, color="steelblue", linewidth=0.5)
axes[1].set_xlabel("Sample Index (sorted by risk)")
axes[1].set_ylabel("Instance Risk")
axes[1].set_title("Sorted Instance Risks (Easy → Hard)")
axes[1].grid(linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("../boia_instance_specific_risks.png", dpi=300, bbox_inches="tight")
plt.show()

## Summary

In [ ]:
print(f"\n{'='*60}")
print(f"BOIA Risk Computation Summary")
print(f"{'='*60}")
print(f"  Model:    {args.model}")
print(f"  Dataset:  {args.dataset}")
print(f"  Seed:     {args.seed}")
print(f"  Device:   {model.device}")
print(f"")
print(f"  Class-specific risks saved to:    {save_path_class}")
print(f"    Shape: {per_concept_risks.shape}")
print(f"    Mean error rate: {overall_concept_error:.4f}")
print(f"")
print(f"  Instance-specific risks saved to: {save_path_instance}")
print(f"    Shape: {instance_risks.shape}")
print(f"    Mean risk: {np.mean(instance_risks):.4f}")
print(f"{'='*60}")